# Treinamento YOLOv8 - Detecção de talheres
Notebook do projeto final para treinar e avaliar um detector de garfo, faca e colher com 100 imagens sintéticas geradas no Blender.


## 1. Instalação e configuração
Instalação da biblioteca Ultralytics e verificação da GPU disponível.


In [ ]:
!pip install -q ultralytics

import torch
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## 2. Carregamento do dataset
Envie `dataset_talheres.zip` para o Colab e descompacte-o em `/content`.


In [ ]:
from google.colab import files
files.upload()
!unzip -q dataset_talheres.zip -d /content/


In [ ]:
from pathlib import Path
base = Path('/content/dataset_talheres')
for divisao in ['train', 'valid', 'test']:
    ni = len(list((base/divisao/'images').glob('*.png')))
    nr = len(list((base/divisao/'labels').glob('*.txt')))
    print(f'{divisao}: {ni} imagens e {nr} rótulos')


## 3. Configuração do dataset
O YAML abaixo usa o caminho válido dentro do ambiente Colab.


In [ ]:
yaml = '''path: /content/dataset_talheres
train: train/images
val: valid/images
test: test/images
names:
  0: garfo
  1: faca
  2: colher
'''
Path('/content/dataset_talheres/dataset.yaml').write_text(yaml, encoding='utf-8')
print(yaml)


## 4. Construção e treinamento
O YOLOv8n pré-treinado é ajustado por 50 épocas, com imagens 640 x 640 e lote de oito imagens.


In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolov8n.pt')
resultado = modelo.train(
    data='/content/dataset_talheres/dataset.yaml',
    epochs=50, imgsz=640, batch=8, device=0,
    project='/content/resultados', name='talheres_yolov8',
    pretrained=True, patience=15, plots=True
)


## 5. Avaliação no conjunto de teste
O melhor peso é carregado para avaliação independente nas dez imagens sintéticas reservadas para teste.


In [ ]:
melhor_modelo = YOLO('/content/resultados/talheres_yolov8/weights/best.pt')
metricas = melhor_modelo.val(
    data='/content/dataset_talheres/dataset.yaml',
    split='test', imgsz=640, device=0, plots=True
)
print(f'Precisão: {metricas.box.mp:.4f}')
print(f'Revocação: {metricas.box.mr:.4f}')
print(f'mAP50: {metricas.box.map50:.4f}')
print(f'mAP50-95: {metricas.box.map:.4f}')


## 6. Predição e visualização
As detecções do conjunto de teste são salvas e uma amostra é apresentada no notebook.


In [ ]:
resultados = melhor_modelo.predict(
    source='/content/dataset_talheres/test/images',
    conf=0.25, imgsz=640, save=True,
    project='/content/predicoes', name='teste'
)

import matplotlib.pyplot as plt
imagem = resultados[0].plot()
plt.figure(figsize=(8, 8))
plt.imshow(imagem[:, :, ::-1])
plt.axis('off');


## 7. Exportação
Compactação dos pesos, métricas, gráficos e imagens previstas.


In [ ]:
!zip -qr /content/resultados_yolo_talheres.zip /content/resultados /content/predicoes
from google.colab import files
files.download('/content/resultados_yolo_talheres.zip')
